In [2]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

In [3]:
filepath = "../data/interim/customer_churn_clean.csv"  
df = pd.read_csv(filepath)

In [22]:
target_col='Churn'
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

def run_categorical_analysis(df, categorical_cols, target_col='Churn'):
    print("--- CATEGORICAL FEATURES VS CHURN (CHI-SQUARE & CRAMER'S V) ---")
    results=[]
    for col in categorical_cols:
        if col == target_col:
            continue
            
        print(f"\nAnalyzing Feature: {col}")
        
        # 1. Contingency Table
        contingency_table = pd.crosstab(df[col], df[target_col])
        print("Contingency Table:")
        print(contingency_table)
        
        # 2. Chi-Square Test
        chi2, p_val, dof, expected = stats.chi2_contingency(contingency_table)
        print(f"Chi-Square Statistic: {chi2:.4f}")
        print(f"P-Value: {p_val:.6f}")
        
        # 3. Cramér's V Calculation
        n = contingency_table.sum().sum()
        r, c = contingency_table.shape
        min_dim = min(r - 1, c - 1)
        
        if min_dim > 0:
            cramer_v = np.sqrt(chi2 / (n * min_dim))
        else:
            cramer_v = 0
            
        print(f"Cramér's V (Effect Size): {cramer_v:.4f}")
        
        # 4. Decision
        if p_val < 0.05:
            print("Decision: Statistically Significant (Strong Association)")
        else:
            print("Decision: Drop Candidate (No Significant Association)")
        print("-" * 40)

        decision = "Statistically Significant" if p_val < 0.05 else "Drop Candidate"
        
        results.append({
            'Feature': col, 'Test Type': 'Chi-Square', 'Test Statistic': round(chi2, 4),
            'P-Value': round(p_val, 6) if p_val > 0.000001 else "< 0.000001",
            'Effect Size': round(cramer_v, 4), 'Decision': decision
        })
    return results
categorical_results = run_categorical_analysis(df, categorical_cols, target_col='Churn')

C:\Users\Administrator\AppData\Local\Temp\ipykernel_1616\1475823079.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()


--- CATEGORICAL FEATURES VS CHURN (CHI-SQUARE & CRAMER'S V) ---

Analyzing Feature: gender
Contingency Table:
Churn      0    1
gender           
Female  2549  939
Male    2625  930
Chi-Square Statistic: 0.4841
P-Value: 0.486579
Cramér's V (Effect Size): 0.0083
Decision: Drop Candidate (No Significant Association)
----------------------------------------

Analyzing Feature: Partner
Contingency Table:
Churn       0     1
Partner            
No       2441  1200
Yes      2733   669
Chi-Square Statistic: 158.7334
P-Value: 0.000000
Cramér's V (Effect Size): 0.1501
Decision: Statistically Significant (Strong Association)
----------------------------------------

Analyzing Feature: Dependents
Contingency Table:
Churn          0     1
Dependents            
No          3390  1543
Yes         1784   326
Chi-Square Statistic: 189.1292
P-Value: 0.000000
Cramér's V (Effect Size): 0.1639
Decision: Statistically Significant (Strong Association)
----------------------------------------

Analyzing Fea

Churn            0     1
PhoneService            
No             512   170
Yes           4662  1699
Chi-Square Statistic: 0.9150
P-Value: 0.338783
Cramér's V (Effect Size): 0.0114
Decision: Drop Candidate (No Significant Association)
----------------------------------------

Analyzing Feature: MultipleLines
Contingency Table:
Churn             0     1
MultipleLines            
No             3053  1019
Yes            2121   850
Chi-Square Statistic: 11.1433
P-Value: 0.000843
Cramér's V (Effect Size): 0.0398
Decision: Statistically Significant (Strong Association)
----------------------------------------

Analyzing Feature: InternetService
Contingency Table:
Churn               0     1
InternetService            
DSL              1962   459
Fiber optic      1799  1297
No               1413   113
Chi-Square Statistic: 732.3096
P-Value: 0.000000
Cramér's V (Effect Size): 0.3225
Decision: Statistically Significant (Strong Association)
----------------------------------------

Analyzing Fea

In [16]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [ ]:
def run_numerical_analysis(df, numerical_cols, target_col='Churn'):
    print("--- NUMERICAL FEATURES VS CHURN (HYPOTHESIS TESTING) ---")
    results=[]
    retained = df[df[target_col] == 0]
    churned = df[df[target_col] == 1]
    
    for col in numerical_cols:
        print(f"\nAnalyzing Feature: {col}")
        
        feat_retained = retained[col].dropna()
        feat_churned = churned[col].dropna()
        
        # 1. Normality Check (Shapiro-Wilk Test - Sampled up to 5000 for library limit)
        _, p_norm = stats.shapiro(df[col].dropna()[:5000])
        is_normal = p_norm > 0.05
        print(f"Normality Test P-Value: {p_norm:.6f} -> Distribution is {'Normal' if is_normal else 'Non-Normal'}")
        
        # 2. Statistical Test based on Distribution
        if not is_normal:
            # Mann-Whitney U Test (Non-Normal)
            stat, p_val = stats.mannwhitneyu(feat_retained, feat_churned, alternative='two-sided')
            print(f"Test Used: Mann-Whitney U Test")
            test_name = 'Mann-Whitney U'
            print(f"Test Statistic: {stat:.4f}")
            print(f"P-Value: {p_val:.6f}")
            
            # Effect Size (Rank-Biserial Correlation)
            n1, n2 = len(feat_retained), len(feat_churned)
            effect_size = abs(1 - (2 * stat) / (n1 * n2)) if (n1 * n2) != 0 else 0
            print(f"Effect Size (Rank-Biserial): {effect_size:.4f}")
            
        else:
            # Independent Two-Sample T-Test (Normal)
            stat, p_val = stats.ttest_ind(feat_retained, feat_churned, equal_var=False)
            test_name = 'Independent T-Test'
            print(f"Test Used: Independent Two-Sample T-Test")
            print(f"Test Statistic: {stat:.4f}")
            print(f"P-Value: {p_val:.6f}")
            
            # Effect Size (Cohen's d)
            v1, v2 = np.var(feat_retained, ddof=1), np.var(feat_churned, ddof=1)
            n1, n2 = len(feat_retained), len(feat_churned)
            pooled_sd = np.sqrt(((n1 - 1) * v1 + (n2 - 1) * v2) / (n1 + n2 - 2))
            effect_size = abs((np.mean(feat_retained) - np.mean(feat_churned)) / pooled_sd) if pooled_sd != 0 else 0
            print(f"Effect Size (Cohen's d): {effect_size:.4f}")
            
        # 3. Decision
            if p_val < 0.05:
                print("Decision: Statistically Significant Difference")
            else:
                print("Decision: Drop Candidate (No Significant Difference)")
            print("-" * 40)
            results.append({
                'Feature': col, 'Test Type': test_name, 'Test Statistic': round(stat, 4),
                'P-Value': round(p_val, 6) if p_val > 0.000001 else "< 0.000001",
                'Effect Size': round(effect_size, 4), 'Decision': decision
            })
        return results

numerical_results=run_numerical_analysis(df, numerical_cols,target_col='Churn')

--- NUMERICAL FEATURES VS CHURN (HYPOTHESIS TESTING) ---

Analyzing Feature: SeniorCitizen
Normality Test P-Value: 0.000000 -> Distribution is Non-Normal
Test Used: Mann-Whitney U Test
Test Statistic: 4226068.0000
P-Value: 0.000000
Effect Size (Rank-Biserial): 0.1260


In [25]:
def run_vif_analysis(df, numerical_cols):
    print("--- MULTICOLLINEARITY ANALYSIS (VIF) ---")
    
    # Missing values remove karna zaroori hai VIF ke liye
    num_df_clean = df[numerical_cols].dropna()
    
    # Intercept constant add karna required hota hai
    X = add_constant(num_df_clean)
    
    vif_data = pd.DataFrame()
    vif_data["Feature"] = numerical_cols
    
    # VIF calculations (index 0 constant ka hota hai, isliye i+1 kiya hai)
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])]
    
    # Rule of thumb interpretations
    vif_data["Status"] = vif_data["VIF"].apply(lambda x: "High Multicollinearity (Drop Candidate)" if x > 5.0 else "Safe / Low Multicollinearity")
    
    print(vif_data.to_string(index=False))
    print("-" * 40)
    return vif_data
run_vif_analysis(df,numerical_cols)

--- MULTICOLLINEARITY ANALYSIS (VIF) ---
       Feature      VIF                                  Status
 SeniorCitizen 1.065487            Safe / Low Multicollinearity
        tenure 5.871072 High Multicollinearity (Drop Candidate)
MonthlyCharges 3.577865            Safe / Low Multicollinearity
  TotalCharges 9.712586 High Multicollinearity (Drop Candidate)
         Churn 1.301746            Safe / Low Multicollinearity
----------------------------------------


,Feature,VIF,Status
0,SeniorCitizen,1.065487,Safe / Low Multicollinearity
1,tenure,5.871072,High Multicollinearity (Drop Candidate)
2,MonthlyCharges,3.577865,Safe / Low Multicollinearity
3,TotalCharges,9.712586,High Multicollinearity (Drop Candidate)
4,Churn,1.301746,Safe / Low Multicollinearity


In [26]:
def create_summary_table(categorical_results, numerical_results):
    print("--- FINAL STATISTICAL SIGNIFICANCE SUMMARY TABLE ---")
    
    # Yeh code aapke pure metrics ko combine karke structured rows generate karega
    # Aap is output ko direct variables se pass karke final result dekh sakte hain
    all_features_data = categorical_results + numerical_results
    
    summary_df = pd.DataFrame(all_features_data, columns=[
        'Feature', 'Test Type', 'Test Statistic', 'P-Value', 'Effect Size', 'Decision'
    ])
    
    return summary_df
create_summary_table(categorical_results,numerical_results)

--- FINAL STATISTICAL SIGNIFICANCE SUMMARY TABLE ---


,Feature,Test Type,Test Statistic,P-Value,Effect Size,Decision
0,gender,Chi-Square,0.4841,0.486579,0.0083,Drop Candidate
1,Partner,Chi-Square,158.7334,< 0.000001,0.1501,Statistically Significant
2,Dependents,Chi-Square,189.1292,< 0.000001,0.1639,Statistically Significant
3,PhoneService,Chi-Square,0.9150,0.338783,0.0114,Drop Candidate
4,MultipleLines,Chi-Square,11.1433,0.000843,0.0398,Statistically Significant
5,InternetService,Chi-Square,732.3096,< 0.000001,0.3225,Statistically Significant
6,OnlineSecurity,Chi-Square,205.6331,< 0.000001,0.1709,Statistically Significant
7,OnlineBackup,Chi-Square,47.2609,< 0.000001,0.0819,Statistically Significant
8,DeviceProtection,Chi-Square,30.5134,< 0.000001,0.0658,Statistically Significant
9,TechSupport,Chi-Square,190.1668,< 0.000001,0.1643,Statistically Significant
